# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| | |
|---|---|
| **Name** | _[your full name]_ |
| **Student ID** | _[your ID]_ |
| **Section** | _[your section]_ |
| **Date submitted** | _[date]_ |

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
%pip install pandas polars numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [3]:
DATA_PATH = "../data/organizations-100000.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (100000, 9)
polars shape: (100000, 9)

columns: ['Index', 'Organization Id', 'Name', 'Website', 'Country', 'Description', 'Founded', 'Industry', 'Number of employees']

   Index  Organization Id            Name                       Website  \
0      1  8cC6B5992C0309c     Acevedo LLC      https://www.donovan.com/   
1      2  ec094061FeaF7Bc  Walls-Mcdonald      http://arias-willis.net/   
2      3  DAcC5dbc58946A7     Gregory PLC  http://www.lynch-hoover.net/   

                         Country                           Description  \
0  Holy See (Vatican City State)      Multi-channeled bottom-line core   
1                      Lithuania     Compatible encompassing groupware   
2                        Tokelau  Multi-channeled intangible help-desk   

   Founded                     Industry  Number of employees  
0     2019  Graphic Design / Web Design                 7070  
1     2005                    Utilities                 8156  
2     2019             Leisure

---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [4]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [5]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.042 ms   <- nothing was executed
lazy WITH .collect():      13.860 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` |   0.042 ms|
| Lazy query with `.collect()` | 13.860 ms |

In one sentence — why is the first number meaningless?

because we are timing the time to write the plan and not the execute time of the query 

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [6]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "Industry"    # a text/category column, for grouping
NUMERIC_COL  = "Founded"     # a numeric column
NUMERIC_COL2 = "Number of employees"  # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] Industry
[ok] Founded
[ok] Number of employees


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [7]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:   247.50 ms
polars:    13.81 ms
ratio:     17.93x


**Observation:** _[one or two sentences — what did you see?]_

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [8]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 1996.0
pandas:     9.24 ms
polars:     1.34 ms
ratio:      6.87x


**Observation:** polars read the dataset approximately 14× faster than Pandas

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [9]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     0.56 ms
polars:     0.27 ms
ratio:      2.07x


**Observation:** pandas took about 0.58× the time of Polars

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [10]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    13.85 ms
polars:     4.18 ms
ratio:      3.32x

pandas result (first rows):
Industry
Accounting                        1366986
Airlines / Aviation               1333474
Alternative Dispute Resolution    1407831
Alternative Medicine              1356758
Animation                         1408785
Name: Founded, dtype: int64

polars result (first rows):
shape: (5, 2)
┌───────────────────────────────┬─────────┐
│ Industry                      ┆ Founded │
│ ---                           ┆ ---     │
│ str                           ┆ i64     │
╞═══════════════════════════════╪═════════╡
│ Graphic Design / Web Design   ┆ 1410958 │
│ Human Resources / HR          ┆ 1272899 │
│ Wireless                      ┆ 1302849 │
│ Primary / Secondary Education ┆ 1323507 │
│ Semiconductors                ┆ 1426351 │
└───────────────────────────────┴─────────┘


**Observation:** _[Polars executed the group-by aggregation faster than Pandas due to its multi-threaded query engine.]_

**Do both libraries give the same numbers?** _[Yes, both libraries produce the exact same aggregate values (sum and mean), though the row order differs because Polars does not sort group keys by default.]_

## 4.5 — Sorting and taking the top N

In [11]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:    32.41 ms
polars:     6.30 ms
ratio:      5.14x


**Observation:** _[Polars sorted the dataset faster than Pandas by using multi-threading and optimized array operations.]_

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [12]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):   278.07 ms
polars (lazy + collect):    13.02 ms
ratio:    21.36x


**Look at the query plan.** Polars can show you what it decided to do.

In [13]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("Founded").sum()] BY [col("Industry")]
  FROM
  simple π 2/2 ["Industry", "Founded"]
    Csv SCAN [../data/organizations-100000.csv]
    PROJECT 2/9 COLUMNS
    SELECTION: col("Founded").cast(Float64) > 1996.0
    ESTIMATED ROWS: 89282


**Observation:** _[your words — what does the plan show? Can you see
anything in it about which columns are read, or where the filter is
applied?]_

The explain plan shows query optimizations like projection pushdown (reading only required columns) and predicate pushdown (filtering early before processing).

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [14]:
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    11.71 ms
polars:     1.66 ms
ratio:      7.04x

rows before: 100000 -> after: 100000
row count unchanged: True


**Observation:** _[Polars completed the hash-join faster than Pandas while keeping the original row count unchanged.]_

---
# Part 5 · Results summary

Collect every measurement into one table.

In [15]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,247.50,13.81,17.93,polars
1,filter,9.24,1.34,6.87,polars
2,new column,0.56,0.27,2.07,polars
3,group by,13.85,4.18,3.32,polars
4,sort + top N,32.41,6.30,5.14,polars
5,chained pipeline,278.07,13.02,21.36,polars
6,join,11.71,1.66,7.04,polars


In [16]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.22631-SP0
   Intel64 Family 6 Model 165 Stepping 3, GenuineIntel


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

The biggest difference was in reading the file. Pandas took 403.64 ms, while Polars took 22.46 ms, making Polars about 18x faster. The full pipeline was also a big difference (429.37 vs. 26.21 ms, 16.4x), but it includes reading the CSV file, so the file reading time explains most of the gap. The smallest Polars improvement was in group by (17.26 vs. 6.43 ms, 2.69x). For creating a new column, pandas was actually faster (0.43 vs. 0.75 ms).
Our theory is that reading a CSV mainly involves parsing text, and Polars does this using Rust and multiple CPU cores, while pandas mostly uses one core. For group by, our data only has 147 industries, so there is less work to split across cores. Creating a new column only involves one subtraction and takes less than a millisecond in both libraries, so the overhead matters more than the actual operation.
### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

We expected Polars to be faster in all operations, but pandas was actually faster at creating a new column. We also noticed that the results changed between runs. For example, pandas group by took 27.19 ms in one run and 17.26 ms in our final table, even though we did not change the code.

When checking the group by results, the totals matched, but the row order was different because Polars does not automatically sort the groups. At first, this made the result look incorrect. We also noticed that the query plan showed col("Founded").cast(Float64) > 1996.0. This happened because our threshold came from .median(), which returns a float, so Polars converts the integer column before comparing it.

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

Pandas felt easier for simple operations like df[df[col] > x] because it is what we learned first. However, Polars was clearer when writing longer pipelines because the steps are easier to follow from top to bottom using pl.col(...) and .alias(...), instead of nested brackets.

We also had to look up some differences in the syntax, such as group_by instead of groupby, descending=True instead of ascending=False, and with_columns instead of assign. The lazy API also took some getting used to, especially remembering to use .collect() to get the results.

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

Mostly, yes. Reading the file (18x), the full pipeline (16x), filtering (7.9x), sorting (6.1x), and joining (5.7x) were all more than 5x faster with Polars. However, group by was only 2.7x faster, and pandas performed better when creating a new column.

Published benchmarks usually use files that are several GB in size and machines with many CPU cores, where parallel processing makes a bigger difference. Our dataset has 100,000 rows and is about 14 MB, so some operations only take a few milliseconds, making overhead a bigger part of the total time.

The results also depend on the operations being tested, whether the file is already cached, and the pandas version. We used pandas 3.0, which stores text columns with Arrow by default, making it faster than older pandas versions used in many published comparisons.

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

I would recommend pandas for small or medium-sized datasets, such as files under a few hundred MB, especially when the project already uses libraries like scikit-learn or seaborn that work with pandas DataFrames. For example, if we are cleaning a 50,000-row survey CSV and plotting the results, saving a few milliseconds would not make much difference, especially if the whole team already knows pandas.

I would recommend Polars for larger files or pipelines that run repeatedly. For example, a daily job that reads a multi-GB log or sales CSV, filters the data, and groups it by category would benefit from Polars because its lazy API can read only the required columns and use multiple CPU cores.

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

Modin (https://modin.readthedocs.io/) is another library worth looking at. It is designed as a drop-in replacement for pandas, meaning we can change import pandas as pd to import modin.pandas as pd and keep the same code.

The main idea behind Modin is to make pandas operations run in parallel instead of relying mostly on one CPU core. It splits the DataFrame and uses frameworks like Ray or Dask to run operations in parallel, without requiring users to learn a completely new API.

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot